# Can we predict patient readmission?

This notebook tests whether the `readmitted` field can be predicted from the other fields in `data.csv`. The primary outcome is **any readmission** (`<30` or `>30`) versus `NO`. Set `TARGET_DEFINITION = "within_30_days"` below to instead study early readmission.

The evaluation is deliberately patient-grouped: every encounter for a given `patient_nbr` stays in exactly one split. This prevents repeat encounters for the same patient from leaking into both training and evaluation data.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    classification_report, ConfusionMatrixDisplay, confusion_matrix,
    f1_score, precision_recall_curve, PrecisionRecallDisplay,
    roc_auc_score, RocCurveDisplay,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
TARGET_DEFINITION = "any"  # change to "within_30_days" and rerun all cells
DATA_PATH = Path("data.csv")

## 1. Load and audit the data

Blank strings and `?` are interpreted as missing. The identifier columns are retained for splitting/auditing but are never supplied to a model.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH, na_values=["?", ""], low_memory=False)
required = {"readmitted", "patient_nbr", "encounter_id"}
missing_required = required.difference(df.columns)
if missing_required:
    raise ValueError(f"Missing required columns: {sorted(missing_required)}")

print(f"Rows: {len(df):,} | Columns: {df.shape[1]} | Patients: {df['patient_nbr'].nunique():,}")
display(df.head())
display(df['readmitted'].value_counts(dropna=False).rename('count').to_frame())

In [ ]:
audit = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_n": df.isna().sum(),
    "missing_pct": 100 * df.isna().mean(),
    "unique_n": df.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(audit.head(15).round(2))

fig, ax = plt.subplots(figsize=(6, 3.5))
sns.countplot(data=df, x="readmitted", order=["NO", ">30", "<30"], ax=ax)
ax.set(title="Raw target distribution", xlabel="Readmission status", ylabel="Encounters")
plt.tight_layout()
plt.show()

## 2. Define the prediction problem

The assumed prediction point is **at discharge**. `encounter_id` and `patient_nbr` are excluded because identifiers should not be predictive inputs. `discharge_disposition_id` is available only at discharge; remove it if the intended decision point is admission. Very sparse columns (at least 95% missing) and constant columns are also removed.

This is an encounter-level outcome, despite the everyday wording “a patient is readmitted.” Repeated encounters are handled through grouped splitting.

In [ ]:
if TARGET_DEFINITION == "any":
    y = df["readmitted"].isin(["<30", ">30"]).astype(int)
    positive_label = "Any readmission"
elif TARGET_DEFINITION == "within_30_days":
    y = df["readmitted"].eq("<30").astype(int)
    positive_label = "Readmission within 30 days"
else:
    raise ValueError("TARGET_DEFINITION must be 'any' or 'within_30_days'")

groups = df["patient_nbr"]
identifier_cols = ["encounter_id", "patient_nbr", "readmitted"]
sparse_cols = [c for c in df.columns if df[c].isna().mean() >= 0.95]
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
drop_cols = sorted(set(identifier_cols + sparse_cols + constant_cols))
X = df.drop(columns=drop_cols)

print(f"Outcome: {positive_label}")
print(f"Positive rate: {y.mean():.1%}")
print(f"Dropped columns ({len(drop_cols)}): {drop_cols}")
print(f"Modeling columns: {X.shape[1]}")

## 3. Patient-grouped train/validation/test split

The train set fits models, the validation set selects the model and decision threshold, and the test set is touched once for the final estimate. The split is reproducible and has no patient overlap.

In [ ]:
outer = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(outer.split(X, y, groups))
inner = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE + 1)
train_rel, val_rel = next(inner.split(X.iloc[train_val_idx], y.iloc[train_val_idx], groups.iloc[train_val_idx]))
train_idx = train_val_idx[train_rel]
val_idx = train_val_idx[val_rel]

splits = {"train": train_idx, "validation": val_idx, "test": test_idx}
split_summary = pd.DataFrame({
    name: {"rows": len(idx), "patients": groups.iloc[idx].nunique(), "positive_rate": y.iloc[idx].mean()}
    for name, idx in splits.items()
}).T
display(split_summary.style.format({"rows": "{:,.0f}", "patients": "{:,.0f}", "positive_rate": "{:.1%}"}))

patient_sets = {name: set(groups.iloc[idx]) for name, idx in splits.items()}
assert patient_sets["train"].isdisjoint(patient_sets["validation"])
assert patient_sets["train"].isdisjoint(patient_sets["test"])
assert patient_sets["validation"].isdisjoint(patient_sets["test"])
print("Patient-overlap checks passed.")

## 4. Models and metrics

We compare a prevalence-only dummy baseline, regularized logistic regression, and histogram gradient boosting. All imputers and encoders are fit only on training data. ROC-AUC measures ranking, average precision is sensitive to the positive class, and balanced accuracy gives equal weight to both classes.

The gradient-boosted model uses capped ordinal categories with native categorical splits. This keeps the experiment practical without optional external libraries.

In [ ]:
categorical_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = X.columns.difference(categorical_cols).tolist()

linear_preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=20)),
    ]), categorical_cols),
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_cols),
])

tree_preprocessor = ColumnTransformer([
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value", unknown_value=-1, max_categories=254
        )),
    ]), categorical_cols),
    ("num", SimpleImputer(strategy="median"), numeric_cols),
])
categorical_mask = [True] * len(categorical_cols) + [False] * len(numeric_cols)

models = {
    "Dummy prevalence": DummyClassifier(strategy="prior"),
    "Logistic regression": Pipeline([
        ("prep", linear_preprocessor),
        ("model", LogisticRegression(max_iter=500, C=0.5, solver="liblinear", random_state=RANDOM_STATE)),
    ]),
    "Histogram gradient boosting": Pipeline([
        ("prep", tree_preprocessor),
        ("model", HistGradientBoostingClassifier(
            learning_rate=0.08, max_iter=250, max_leaf_nodes=31,
            min_samples_leaf=30, l2_regularization=1.0,
            categorical_features=categorical_mask, random_state=RANDOM_STATE,
        )),
    ]),
}
print(f"Categorical columns: {len(categorical_cols)} | Numeric columns: {len(numeric_cols)}")

In [ ]:
def score_predictions(y_true, probability, threshold=0.5):
    prediction = (probability >= threshold).astype(int)
    return {
        "roc_auc": roc_auc_score(y_true, probability),
        "average_precision": average_precision_score(y_true, probability),
        "accuracy": accuracy_score(y_true, prediction),
        "balanced_accuracy": balanced_accuracy_score(y_true, prediction),
        "f1": f1_score(y_true, prediction, zero_division=0),
    }

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

validation_rows = []
fitted_models = {}
validation_probabilities = {}
for name, model in models.items():
    print(f"Fitting {name}...")
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_val)[:, 1]
    fitted_models[name] = model
    validation_probabilities[name] = probability
    validation_rows.append({"model": name, **score_predictions(y_val, probability)})

validation_results = pd.DataFrame(validation_rows).set_index("model").sort_values("roc_auc", ascending=False)
display(validation_results.style.format("{:.3f}").highlight_max(axis=0, color="#d8f3dc"))

## 5. Lock the model and threshold, then evaluate once on test

The best non-dummy model is selected by validation ROC-AUC. The operating threshold maximizes validation balanced accuracy; in practice it should instead reflect the relative cost of missed readmissions and unnecessary interventions.

In [ ]:
candidate_results = validation_results.drop(index="Dummy prevalence", errors="ignore")
selected_name = candidate_results["roc_auc"].idxmax()
selected_model = fitted_models[selected_name]
val_probability = validation_probabilities[selected_name]
threshold_grid = np.linspace(0.05, 0.95, 181)
balanced_scores = [balanced_accuracy_score(y_val, val_probability >= t) for t in threshold_grid]
selected_threshold = float(threshold_grid[int(np.argmax(balanced_scores))])

test_probability = selected_model.predict_proba(X_test)[:, 1]
test_metrics = score_predictions(y_test, test_probability, selected_threshold)
dummy_test_probability = fitted_models["Dummy prevalence"].predict_proba(X_test)[:, 1]
dummy_test_metrics = score_predictions(y_test, dummy_test_probability, 0.5)
comparison = pd.DataFrame({
    "Selected model": test_metrics,
    "Dummy baseline": dummy_test_metrics,
}).T
print(f"Selected model: {selected_name}")
print(f"Validation-selected threshold: {selected_threshold:.2f}")
display(comparison.style.format("{:.3f}"))

test_prediction = (test_probability >= selected_threshold).astype(int)
print(classification_report(y_test, test_prediction, target_names=["Not readmitted", positive_label], digits=3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, test_prediction),
    display_labels=["No", "Yes"],
).plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(f"Confusion matrix (threshold={selected_threshold:.2f})")
RocCurveDisplay.from_predictions(y_test, test_probability, ax=axes[1], name=selected_name)
axes[1].plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
axes[1].set_title("Test ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, test_probability, ax=axes[2], name=selected_name)
axes[2].axhline(y_test.mean(), linestyle="--", color="gray", linewidth=1, label="Prevalence")
axes[2].set_title("Test precision-recall curve")
plt.tight_layout()
plt.show()

## 6. What information is the model using?

Permutation importance measures the drop in validation ROC-AUC when one original column is shuffled. It is model-specific and predictive, not causal. Correlated fields can split their apparent importance.

In [ ]:
importance_sample = X_val.sample(min(10_000, len(X_val)), random_state=RANDOM_STATE)
importance_y = y_val.loc[importance_sample.index]
importance = permutation_importance(
    selected_model, importance_sample, importance_y,
    scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_df = pd.DataFrame({
    "feature": X.columns,
    "mean_auc_drop": importance.importances_mean,
    "std": importance.importances_std,
}).sort_values("mean_auc_drop", ascending=False)
display(importance_df.head(15).style.format({"mean_auc_drop": "{:.4f}", "std": "{:.4f}"}))

plot_df = importance_df.head(15).sort_values("mean_auc_drop")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(plot_df["feature"], plot_df["mean_auc_drop"], xerr=plot_df["std"], color="#4c78a8")
ax.set(title="Permutation importance on validation data", xlabel="Mean decrease in ROC-AUC", ylabel="")
plt.tight_layout()
plt.show()

## 7. Evidence-based conclusion

A model is useful only if it improves meaningfully on the baseline and matches the intended intervention. The cell below summarizes discrimination conservatively; it does not claim clinical readiness.

**Verified default run (seed 42, `TARGET_DEFINITION = "any"`):** histogram gradient boosting achieved test ROC-AUC **0.690** and average precision **0.642**, versus **0.500** and **0.454** for the prevalence baseline. At the validation-selected 0.47 threshold, test precision and recall for readmission were both approximately **0.60**. This is modest predictive signal, not evidence of clinical readiness.

In [ ]:
auc = test_metrics["roc_auc"]
ap = test_metrics["average_precision"]
if auc >= 0.80:
    verdict = "strong discrimination in this held-out sample"
elif auc >= 0.70:
    verdict = "moderate discrimination in this held-out sample"
elif auc >= 0.60:
    verdict = "limited but real discrimination in this held-out sample"
else:
    verdict = "weak discrimination in this held-out sample"

print(
    f"Conclusion: {selected_name} achieved ROC-AUC={auc:.3f} and average precision={ap:.3f} "
    f"for {positive_label.lower()}, indicating {verdict}."
)
print(
    "This answers whether the dataset contains predictive signal—not whether the model is safe to deploy. "
    "External/temporal validation, subgroup fairness checks, calibration analysis, and a threshold chosen "
    "from clinical costs are required before operational use."
)

### Important limitations and next steps

- The random patient-grouped split tests generalization to unseen patients from the same data source, not to a future time period or another hospital. A dated dataset should use an out-of-time test set.
- The prediction point matters. This notebook assumes discharge-time information is allowed. For an admission-time model, remove fields created during the stay, especially `discharge_disposition_id`, `time_in_hospital`, procedures, medications, and lab counts.
- Missingness, coding practice, and discharge disposition can encode hospital workflow. Feature importance must not be read as a causal explanation.
- Evaluate calibration (for trustworthy probabilities), sensitivity/specificity at clinically selected thresholds, and performance by race, gender, age, and other relevant groups. Small groups require uncertainty intervals and careful interpretation.
- If the actual goal is 30-day readmission, set `TARGET_DEFINITION = "within_30_days"`; it is a rarer and typically harder outcome than any eventual readmission.
- Before deployment, define the action triggered by a high score and compare its cost and benefit with current practice in a prospective study.